# QAT — Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference (2017)

_Papers · Efficiency & Compression_

**Map real numbers to int8 with one affine rule, simulate that rounding while training, and the net stays accurate after you throw away the floats.**

---

This notebook builds quantization-aware training one step at a time: the affine int8 map, a fake-quant op with a straight-through gradient, a net that simulates rounding as it trains, and an ablation that breaks the gradient. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — The affine quantization map by hand

Integer inference rests on one affine rule: a real number $r$ is represented by an integer $q$ via $r = S(q - Z)$, where $S$ is the **scale** (step size) and $Z$ is the **zero-point** (the integer that maps to real $0$). To quantize we invert that — divide by $S$, round to the grid, and clamp into range — then dequantize maps back. We check the rule on one value so the rounding and the small error are concrete.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# Worked example: affine map r = S(q - Z), uint8 (n=256).
a, b = -2.0, 6.0
S = (b - a) / 255.0                      # scale = step size  (= 8/255)
Z = round(0.0 - a / S)                   # zero-point: the integer that maps to real 0

r = 1.5
q = max(0, min(255, round(r / S) + Z))   # quantize: invert r = S(q - Z), then clamp
r_hat = S * (q - Z)                       # dequantize

print(f"worked example:  S = {S:.6f} (=8/255),  Z = {Z}")
print(f"  quantize r=1.5 -> q = {q};  dequantize -> {r_hat:.6f};  err = {r - r_hat:+.6f}")
# worked example:  S = 0.031373 (=8/255),  Z = 64
#   quantize r=1.5 -> q = 112;  dequantize -> 1.505882;  err = -0.005882

### Step 2 — Fake-quant with a straight-through gradient, and the quantized net

`FakeQuant` rounds a tensor onto the int grid in the **forward** pass (so the net feels int8 error) but passes the gradient through unchanged in the **backward** pass — the straight-through estimator (STE). Round's true derivative is zero almost everywhere, so without the STE no gradient would reach the weights. `Net` is a 2-hidden-layer classifier that, when `quant=True`, fake-quants its weights (range from each tensor's own min/max) and its activations (range tracked by an EMA of the max). The bias stays in float.

In [ ]:
# --- Affine fake-quant (Eq. 12) with the straight-through estimator. ---
class FakeQuant(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, a, b, nbits):
        qmax = 2 ** nbits - 1
        S = (b - a) / qmax                       # step size
        q = torch.clamp(torch.round((x - a) / S), 0, qmax)   # round onto the int grid
        return a + q * S                         # map back to a real value (fake-quant)
    @staticmethod
    def backward(ctx, grad):
        return grad, None, None, None            # STE: pass gradient straight through

def fq(x, a, b, nbits=8):
    return FakeQuant.apply(x, a, b, nbits)

# --- A tiny 2-hidden-layer classifier; fake-quant weights + activations when quant=True. ---
class Net(nn.Module):
    def __init__(self, n, K):
        super().__init__()
        self.fc1 = nn.Linear(n, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, K)
        self.e1 = None                           # EMA of fc1 activation max
        self.e2 = None                           # EMA of fc2 activation max
    def forward(self, x, quant=False, nbits=8):
        if not quant:
            h = torch.relu(self.fc1(x))
            h = torch.relu(self.fc2(h))
            return self.fc3(h)
        # weights: range = the tensor's own min/max
        def qw(L):
            return fq(L.weight, L.weight.min().item(), L.weight.max().item(), nbits)
        h = torch.relu(F.linear(x, qw(self.fc1), self.fc1.bias))    # bias kept in float
        m = h.max().item()                                          # activation range via EMA
        if self.training:
            self.e1 = m if self.e1 is None else 0.95 * self.e1 + 0.05 * m
        h = fq(h, 0.0, self.e1 or m, nbits)
        h = torch.relu(F.linear(h, qw(self.fc2), self.fc2.bias))
        m = h.max().item()
        if self.training:
            self.e2 = m if self.e2 is None else 0.95 * self.e2 + 0.05 * m
        h = fq(h, 0.0, self.e2 or m, nbits)
        return F.linear(h, qw(self.fc3), self.fc3.bias)

### Step 3 — Train QAT vs non-QAT, then sweep the bit-width

The task is 6 overlapping Gaussian blobs in 12-D — close enough that coarse bits actually hurt. We train two nets to a similar float baseline: **Net A** in plain float (quantized only afterward = post-training quantization, PTQ) and **Net B** quantization-aware (it sees the rounding every forward pass = QAT). Then we sweep the bit-width down from 8 to 2 and compare. QAT should degrade more gracefully because it trained against the rounding error.

In [ ]:
# --- A harder toy task: 6 overlapping Gaussian blobs in 12-D (so coarse bits actually hurt). ---
N, K, n = 1200, 6, 12
g = torch.Generator().manual_seed(1)
y = torch.randint(0, K, (N,), generator=g)
centers = torch.randn(K, n, generator=g) * 1.2
X = centers[y] + torch.randn(N, n, generator=g) * 1.1     # noisy + close centers
Xtr, ytr, Xte, yte = X[:800], y[:800], X[800:], y[800:]

def accuracy(net, **kw):
    net.eval()
    with torch.no_grad():
        return (net(Xte, **kw).argmax(1) == yte).float().mean().item()

def train(quant_aware, steps=600, lr=0.1):
    torch.manual_seed(0)
    net = Net(n, K)
    net.train()
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    lf = nn.CrossEntropyLoss()
    for t in range(steps):
        opt.zero_grad()
        lf(net(Xtr, quant=quant_aware, nbits=8), ytr).backward()
        opt.step()
    return net

net_A = train(quant_aware=False)     # plain float (then quantized after the fact = PTQ)
net_B = train(quant_aware=True)      # quantization-aware (QAT)

print("\n=== float accuracy (both reach a similar baseline) ===")
print(f"  Net A (non-QAT) float: {accuracy(net_A):.3f}    Net B (QAT) float: {accuracy(net_B):.3f}")

print("\n=== accuracy vs bit-width  (QAT = Net B,  PTQ = Net A quantized after training) ===")
print("  bits |  QAT  |  PTQ")
for nb_ in [8, 6, 4, 3, 2]:
    print(f"   {nb_}   | {accuracy(net_B, quant=True, nbits=nb_):.3f} | {accuracy(net_A, quant=True, nbits=nb_):.3f}")
# Our small run, not the paper's numbers:
#   8 bits: both ~0.92 (int8 keeps accuracy).  As bits shrink, QAT degrades more gracefully than PTQ.

### Step 4 — Ablation: kill the straight-through estimator

To prove the STE is what makes QAT trainable, we swap in a fake-quant whose backward returns **zero** — the honest derivative of round. With no gradient flowing through the rounding, the weights upstream of every quantized op stop updating and the loss barely moves. This isolates the STE as the single thing that lets the float weights learn despite the non-differentiable forward.

In [ ]:
# --- ABLATION: kill the straight-through estimator (zero backward) -> QAT can't learn. ---
class FakeQuantNoSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, a, b, nbits):
        return FakeQuant.forward(ctx, x, a, b, nbits)
    @staticmethod
    def backward(ctx, grad):
        return torch.zeros_like(grad), None, None, None   # honest round derivative = 0

def train_no_ste(steps=600, lr=0.1):
    torch.manual_seed(0)
    net = Net(n, K)
    net.train()
    orig = FakeQuant.apply
    FakeQuant.apply = staticmethod(lambda *a: FakeQuantNoSTE.apply(*a))  # swap in zero backward
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    lf = nn.CrossEntropyLoss()
    first = last = None
    for t in range(steps):
        opt.zero_grad()
        loss = lf(net(Xtr, quant=True, nbits=8), ytr)
        loss.backward()
        opt.step()
        if t == 0:
            first = loss.item()
        last = loss.item()
    FakeQuant.apply = orig
    return first, last

f0, f1 = train_no_ste()
print(f"\n=== STE ablation (zero backward) ===\n  loss start {f0:.3f} -> end {f1:.3f}  (barely moves: no gradient flows through round)")

## Visualize the data & results

_As the integer bit-width shrinks from 8 to 2, does a quantization-aware net (QAT) keep accuracy better than a non-QAT net quantized after training (PTQ)?_

### Step 1 — Re-create fake-quant, the net, and the data (standalone)

This visualization cell stands on its own, rebuilding `FakeQuant`, `Net`, and the 6-blob dataset exactly as defined above. Re-running it gives a clean, reproducible setup before we sweep the bit-width.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Affine fake-quant (Eq. 12) with the straight-through estimator.
class FakeQuant(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, a, b, nbits):
        qmax = 2 ** nbits - 1
        S = (b - a) / qmax
        q = torch.clamp(torch.round((x - a) / S), 0, qmax)
        return a + q * S
    @staticmethod
    def backward(ctx, grad):
        return grad, None, None, None            # STE

def fq(x, a, b, nbits=8):
    return FakeQuant.apply(x, a, b, nbits)

class Net(nn.Module):
    def __init__(self, n, K):
        super().__init__()
        self.fc1 = nn.Linear(n, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, K)
        self.e1 = None
        self.e2 = None
    def forward(self, x, quant=False, nbits=8):
        if not quant:
            h = torch.relu(self.fc1(x))
            h = torch.relu(self.fc2(h))
            return self.fc3(h)
        def qw(L):
            return fq(L.weight, L.weight.min().item(), L.weight.max().item(), nbits)
        h = torch.relu(F.linear(x, qw(self.fc1), self.fc1.bias))
        m = h.max().item()
        if self.training:
            self.e1 = m if self.e1 is None else 0.95*self.e1 + 0.05*m
        h = fq(h, 0.0, self.e1 or m, nbits)
        h = torch.relu(F.linear(h, qw(self.fc2), self.fc2.bias))
        m = h.max().item()
        if self.training:
            self.e2 = m if self.e2 is None else 0.95*self.e2 + 0.05*m
        h = fq(h, 0.0, self.e2 or m, nbits)
        return F.linear(h, qw(self.fc3), self.fc3.bias)

# Noisy 6-class blobs (close centers -> coarse bits hurt).
N, K, n = 1200, 6, 12
g = torch.Generator().manual_seed(1)
y = torch.randint(0, K, (N,), generator=g)
centers = torch.randn(K, n, generator=g) * 1.2
X = centers[y] + torch.randn(N, n, generator=g) * 1.1
Xtr, ytr, Xte, yte = X[:800], y[:800], X[800:], y[800:]

print("data ready:", Xtr.shape[0], "train /", Xte.shape[0], "test")

### Step 2 — Train both nets and compare accuracy across bit-widths

Train Net A (float/PTQ) and Net B (QAT) to a matched float baseline, then read out accuracy at 8, 6, 4, 3, and 2 bits for each. The printed pair of curves is the headline: as the integer budget shrinks, the quantization-aware net holds accuracy better than the one quantized blind after training.

In [ ]:
def acc(net, **kw):
    net.eval()
    with torch.no_grad():
        return (net(Xte, **kw).argmax(1) == yte).float().mean().item()

def train(qa, steps=600, lr=0.1):
    torch.manual_seed(0)
    net = Net(n, K)
    net.train()
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    lf = nn.CrossEntropyLoss()
    for t in range(steps):
        opt.zero_grad()
        lf(net(Xtr, quant=qa, nbits=8), ytr).backward()
        opt.step()
    return net

A = train(False)
B = train(True)
print("float baseline  A:", round(acc(A),3), " B:", round(acc(B),3))
for nb_ in [8, 6, 4, 3, 2]:
    print(f"{nb_}-bit  QAT={acc(B, quant=True, nbits=nb_):.3f}  PTQ={acc(A, quant=True, nbits=nb_):.3f}")
# QAT : 8->0.920 6->0.925 4->0.905 3->0.890 2->0.757
# PTQ : 8->0.920 6->0.923 4->0.890 3->0.860 2->0.710  (our run, not the paper's numbers)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Quantize by hand. A tensor has real range $[a,b] = [-2.0, 6.0]$ and uses unsigned 8-bit
            integers ($q \in 0\ldots255$). Compute the scale $S$ and the zero-point $Z$, then quantize
            $r = 3.0$ to an integer and dequantize it back. What is the rounding error?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Scale: $S = (b-a)/255 = 8/255 = 0.031373$. — _The step size spreads the range $[-2,6]$ across the $255$ gaps between the $256$ integer levels._
- Zero-point: map $a=-2.0$ to integer $0$, so $Z = 0 - a/S = 2.0/0.031373 = 63.75 \to 64$. — _$Z$ is the integer that represents real $0$; rounding $63.75$ gives the integer $64$._
- Quantize: $q = \text{round}(r/S) + Z = \text{round}(3.0/0.031373) + 64 = \text{round}(95.63) + 64 = 96 + 64 = 160$. — _Invert $r = S(q-Z)$ to get $q = \text{round}(r/S)+Z$; $160$ is inside $[0,255]$ so no clamp._
- Dequantize: $\hat{r} = S(q-Z) = 0.031373 \times (160-64) = 0.031373 \times 96 = 3.01176$. — _Apply Eq. 1 directly with the integer $160$._

**Answer:** $S = 8/255 \approx 0.031373$, $Z = 64$. Quantizing $r=3.0$ gives $q = 160$; dequantizing gives
                 $\hat{r} = 3.01176$, a rounding error of about $+0.0118$ &mdash; under one step $S$. The notebook
                 recomputes the same shape of calculation for $r=1.5 \to 112 \to 1.50588$.

</details>

**Problem 2.** The straight-through estimator ablation. You have quantization-aware training working: the
            fake-quant's backward returns the incoming gradient unchanged. Now change
            backward to return $0$ (the honest derivative of round, which is flat).
            What happens to training, and what does that show?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Replace the STE backward with one that returns zeros for the input gradient. — _An honest ablation changes exactly one thing &mdash; how gradient flows through the round &mdash; so any difference is attributable to the STE._
- Retrain and watch the loss: with a zero backward, the gradient reaching the weights through the fake-quant is $0$, so the weights upstream of every quantized op stop updating and the loss stalls near its starting value. — _Rounding is flat almost everywhere; its true derivative is $0$. Without the straight-through approximation, back-propagation has nothing to push the weights with._
- Restore the STE (return grad unchanged) and training resumes normally. — _Pretending round is the identity in backward lets the float weights learn despite the non-differentiable forward._

**Answer:** With a zero backward the quantized layers block the gradient, so the net barely learns &mdash;
                 the loss stays high. That isolates the straight-through estimator as the thing that makes
                 quantization-aware training trainable: the forward must round (to feel int8 error) while the
                 backward must not (to keep gradients flowing). The CODE panel includes this ablation.

</details>

**Problem 3.** Net A is trained in float and quantized to 3 bits afterward (post-training). Net B is trained with
            3-bit fake-quant on (quantization-aware). Both had the same float accuracy. Which do you expect to
            be more accurate at 3 bits, and why &mdash; in terms of where each net's weights ended up?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Picture the decision boundaries. Net A's weights sit wherever float training left them, with no knowledge that a coarse round is coming. — _Post-training quantization rounds once, after the fact; the 3-bit grid is coarse, so the round can shove activations across a boundary._
- Net B saw the 3-bit rounding error in every forward pass and adjusted its weights (via the STE gradient) to keep the loss low after rounding. — _Training with simulated quantization steers the weights into regions that are robust to the round &mdash; the paper's core idea (§3)._
- Conclude Net B should hold accuracy better at 3 bits; the gap widens as bits shrink. — _The coarser the grid, the larger the round error, and the more it helps to have trained against it._

**Answer:** Net B (quantization-aware) should be more accurate at 3 bits. It trained against the
                 rounding, so its weights landed in places robust to the coarse 3-bit grid; Net A was rounded
                 blind. At a generous 8 bits the two are close; as the budget drops to 3 or 2 bits the
                 quantization-aware net pulls ahead &mdash; exactly the contrast the CODEVIZ panel plots.

</details>